# Predicta Semiconductor Test Analytics — Day 5.5 Feature-Engineered Threshold Optimization

**Validation Dataset**: `ml/data/processed/validation.csv` (6,000 records / 12 unseen wafers)  
**Model Feature Set**: Experiment F (23 Features: 16 Raw + 7 Engineered)  
**Model Performance**: `ROC-AUC = 0.9046`, `PR-AUC = 0.6932`  
**Plot Artifact**: `ml/analysis/plots/engineered_model_thresholds.svg`  

> [!IMPORTANT]
> Complete threshold optimization sweep across 13 candidate threshold values (0.20 to 0.80). Test set (`test.csv`) remains 100% locked.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

val_df = pd.read_csv('../data/processed/validation.csv')
THRESHOLDS = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]


Loaded 6,000 validation records.
Evaluated 13 candidate threshold values.


--- 
## Section 1 — Full Threshold Sweep & Screening Burden Table
Includes Precision, Recall, F1, FPR, TP, TN, FP, FN, and Flagged FAIL Rate (% of total production volume flagged for secondary screening).

In [2]:
# Threshold optimization code cell


Thresh   | Acc (%)  | Prec    | Rec (%)  | F1      | FPR (%)  | Flagged %  | TP   | TN    | FP   | FN  
-------------------------------------------------------------------------------------------------------
0.35     | 50.85    | 0.2032  | 90.83    | 0.3320  | 55.36    | 60.13%     | 733  | 2318  | 2875 | 74   
0.45     | 72.18    | 0.3055  | 83.89    | 0.4479  | 29.64    | 36.93%     | 677  | 3654  | 1539 | 130  
0.55     | 85.12    | 0.4671  | 75.59    | 0.5774  | 13.40    | 21.77%     | 610  | 4497  | 696  | 197  


--- 
## Section 2 — Defect-Wise Detection Breakdown Across Candidate Thresholds
Impact of threshold on subtle defect categories (`EQUIPMENT_DRIFT` and `PROCESS_VARIATION`).

In [3]:
# Defect-wise candidate breakdown code cell


Defect Category    | High-Recall (0.35) | Balanced (0.45)  | Low-Alarm (0.55)
-----------------------------------------------------------------------------
HIGH_LEAKAGE       | 91.01%             | 85.39%           | 78.09%          
LOW_VOLTAGE        | 98.37%             | 90.24%           | 86.99%          
TIMING_FAILURE     | 100.00%            | 99.21%           | 86.61%          
THERMAL_ANOMALY    | 99.01%             | 95.05%           | 90.10%          
POWER_ANOMALY      | 97.06%             | 92.16%           | 88.24%          
PROCESS_VARIATION  | 97.78%             | 80.00%           | 68.89%          
EQUIPMENT_DRIFT    | 41.86%             | 30.23%           | 12.79%          


--- 
## Section 3 — Final Preferred Threshold Summary for ML Lead

```text
=========================================================================
PREFERRED OPERATING THRESHOLD: Threshold = 0.45 (Balanced Candidate)
=========================================================================
  - FAIL Recall         : 83.89% (677 / 807 defects caught)
  - False Alarm Rate    : 29.64% (Cuts screening burden down to 36.93%)
  - Precision           : 0.3055
  - F1-Score            : 0.4479
=========================================================================
```